In [3]:
print("Hello World")

Hello World


In [4]:
from dotenv import load_dotenv
load_dotenv()

True

# Preparing RAG Evaluation Dataset

In [6]:
import pandas as pd

# QA
inputs = [
    "For customer-facing applications, which company's models dominate the top rankings?",
    "What percentage of respondents are using RAG in some form?",
    "How often are most respondents updating their models?",
]

outputs = [
    "OpenAI models dominate, with 3 of the top 5 and half of the top 10 most popular models for customer-facing apps.",
    "70% of respondents are using RAG in some form.",
    "More than 50% update their models at least monthly, with 17% doing so weekly.",
]

# Create dataset
qa_pairs = [
    {"question": q, "answer": a}
    for q, a in zip(inputs, outputs)
]
df = pd.DataFrame(qa_pairs)

# Write to CSV
csv_path = r"F:\AI Projects\End-to-End-LLMOPs-Project\data\goldens.csv"
df.to_csv(csv_path, index=False)

print(f"CSV saved successfully: {csv_path}")

CSV saved successfully: F:\AI Projects\End-to-End-LLMOPs-Project\data\goldens.csv


# Preparing RAG Evaluation Dataset in LangSmith

In [9]:
from langsmith import Client

client = Client()
dataset_name = "Golden Dataset for RAG Evaluation"

# Store
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for AgenticAIReport",
)
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

{'example_ids': ['a6210cd1-99eb-47f2-a0d6-13d1db5b3572',
  '79a89c68-4b36-4ef3-89a0-0fb7bdb592d0',
  'ccb12a94-416f-4d0a-a497-24f2523a0554'],
 'count': 3,
 'as_of': '2026-10-11T14:14:12.758193015Z'}

# Creating a LangSmith RAG Evaluation Function

In [ ]:
import sys
sys.path.append(r"F:\AI Projects\End-to-End-LLMOPs-Project")

from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os

# Simple file adapter for local file paths
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name
    
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = "/Users/yashpatil/Developer/AI/YT/Sunny/LLMOps_series/data/The 2025 AI Engineering Report.txt",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """
    Answer questions about the AI Engineering Report using RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the AI Engineering Report text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}
        
        # Check if file exists
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)
        
        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        )
        
        # Build retriever
        ingestor.built_retriver(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )
        
        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"
        
        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index")
        )
        
        # Get answer
        answer = rag.invoke(question, chat_history=[])
        
        return {"answer": answer}
        
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}

In [14]:

import os
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(r"F:\AI Projects\End-to-End-LLMOPs-Project")
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG


class LocalFileAdapter:
    """Adapts a local file for document ingestion."""

    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name

    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = str(
        PROJECT_ROOT / "data" / "The 2025 AI Engineering Report.txt"
    ),
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """Answer AI Engineering Report questions using RAG."""

    try:
        question = inputs.get("question", "").strip()

        if not question:
            return {"answer": "No question provided."}

        file_path = Path(data_path)

        if not file_path.is_file():
            return {"answer": f"Data file not found: {file_path}"}

        ingestor = ChatIngestor(
            temp_base=str(PROJECT_ROOT / "data"),
            faiss_base=str(PROJECT_ROOT / "faiss_index"),
            use_session_dirs=True
        )

        ingestor.built_retriver(
            uploaded_files=[LocalFileAdapter(str(file_path))],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k,
            search_type="mmr",
            fetch_k=20,
            lambda_mult=0.5
        )

        session_id = ingestor.session_id
        index_path = PROJECT_ROOT / "faiss_index" / session_id

        rag = ConversationalRAG(session_id=session_id)

        rag.load_retriever_from_faiss(
            index_path=str(index_path),
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index"),
            search_type="mmr",
            fetch_k=20,
            lambda_mult=0.5
        )

        answer = rag.invoke(question, chat_history=[])

        return {"answer": answer}

    except Exception as e:
        return {"answer": f"Error: {str(e)}"}


In [15]:
# Test the function with a sample question
test_input = {"question": "For customer-facing applications, which company's models dominate the top rankings?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])

{"timestamp": "2026-10-11T15:18:15.230871Z", "level": "info", "event": "Running in LOCAL mode"}
{"keys": ["GOOGLE_API_KEY", "GROQ_API_KEY"], "timestamp": "2026-10-11T15:18:15.235353Z", "level": "info", "event": "API keys loaded"}
{"session_id": "session_20261011_204815_48aaf079", "temp_dir": "F:\\AI Projects\\End-to-End-LLMOPs-Project\\data\\session_20261011_204815_48aaf079", "faiss_dir": "F:\\AI Projects\\End-to-End-LLMOPs-Project\\faiss_index\\session_20261011_204815_48aaf079", "sessionized": true, "timestamp": "2026-10-11T15:18:15.252115Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Report.txt", "saved_as": "F:\\AI Projects\\End-to-End-LLMOPs-Project\\data\\session_20261011_204815_48aaf079\\the_2025_ai_engineering_report_86e432.txt", "timestamp": "2026-10-11T15:18:15.287800Z", "level": "info", "event": "File saved for ingestion"}
{"count": 1, "timestamp": "2026-10-11T15:18:15.292467Z", "level": "info", "event": "Documents loaded"}
{"c

Question: For customer-facing applications, which company's models dominate the top rankings?

Answer: OpenAI models dominate the top rankings for customer‑facing applications.
